In [4]:
from games.nocca_nocca.nocca_nocca import NoccaNocca
from games.nocca_nocca.nocca_nocca_eval import NoccaNoccaEval
from agents.agent_random import RandomAgent
from agents.minimax import MiniMax
from agents.minimax_alphabeta import MiniMaxAlphaBeta
from tqdm import tqdm

In [5]:
game = NoccaNocca(max_steps=150, initial_player=0, seed=1)

In [6]:
agents = {
    game.agents[0]: MiniMaxAlphaBeta(game=game, agent=game.agents[0], depth=2, alphabeta=True),
    game.agents[1]: MiniMaxAlphaBeta(game=game, agent=game.agents[1], depth=2, alphabeta=True),
}

In [7]:
game.reset()
player = agents[game.agent_selection]
action = player.action()
print(f"Action taken by {player.agent} (Depth={player.depth}): {action} in {player.iterations} iterations")

Action taken by Black (Depth=2): 65 in 63 iterations


In [8]:
game.reset()
print(f"Initial Agent: {game.agent_selection}")
while not game.game_over():
    game.render()
    action = agents[game.agent_selection].action()
    print(f"Turn {game.steps} -- Agent {game.agent_selection} plays action {action}")
    game.step(action=action)
game.render()
if game.truncated():
    print("Game was truncated")
for agent in agents:
    print(f"Reward agent {agent}: {game.reward(agent)}")
print(f"The winner is: {game.check_for_winner()}")

Initial Agent: Black
0: ___ ___ ___ ___ ___ 
1: 0__ 0__ 0__ 0__ 0__ 
2: ___ ___ ___ ___ ___ 
3: ___ ___ ___ ___ ___ 
4: ___ ___ ___ ___ ___ 
5: ___ ___ ___ ___ ___ 
6: 1__ 1__ 1__ 1__ 1__ 
7: ___ ___ ___ ___ ___ 
Turn 0 -- Agent Black plays action 50
0: ___ ___ ___ ___ ___ 
1: 00_ ___ 0__ 0__ 0__ 
2: ___ ___ ___ ___ ___ 
3: ___ ___ ___ ___ ___ 
4: ___ ___ ___ ___ ___ 
5: ___ ___ ___ ___ ___ 
6: 1__ 1__ 1__ 1__ 1__ 
7: ___ ___ ___ ___ ___ 
Turn 1 -- Agent White plays action 259
0: ___ ___ ___ ___ ___ 
1: 00_ ___ 0__ 0__ 0__ 
2: ___ ___ ___ ___ ___ 
3: ___ ___ ___ ___ ___ 
4: ___ ___ ___ ___ ___ 
5: ___ ___ ___ ___ ___ 
6: 1__ 1__ ___ 11_ 1__ 
7: ___ ___ ___ ___ ___ 
Turn 2 -- Agent Black plays action 46
0: ___ ___ ___ ___ ___ 
1: 0__ ___ 0__ 0__ 0__ 
2: ___ 0__ ___ ___ ___ 
3: ___ ___ ___ ___ ___ 
4: ___ ___ ___ ___ ___ 
5: ___ ___ ___ ___ ___ 
6: 1__ 1__ ___ 11_ 1__ 
7: ___ ___ ___ ___ ___ 
Turn 3 -- Agent White plays action 247
0: ___ ___ ___ ___ ___ 
1: 0__ ___ 0__ 0__ 0__ 
2: ___ 0_

Para explorar el desempeño de un agente minimax sobre este ambiente es necesario implementar una función de evaluación (NoccaNocca no expone una predefinida como lo hace Ta-Te-Ti).

Implementamos una función de evaluación custom en NoccaNoccaEval con la siguiente heurística:

$V(s) = alpha * D(s)$

Donde:

$D(s) = distance_p(s) - distance_{1-p}(s)$

$distance_p$ es la distancia entre el final del tablero y la ficha más cercana no bloqueada de $p$. No bloqueada quiere decir que no tiene ninguna ficha del oponente por encima.

In [10]:
game = NoccaNoccaEval(max_steps=150, initial_player=0, seed=1, alpha=-1)

agents = {
    game.agents[0]: MiniMaxAlphaBeta(game=game, agent=game.agents[0], depth=1, alphabeta=True),
    game.agents[1]: MiniMaxAlphaBeta(game=game, agent=game.agents[1], depth=4, alphabeta=True),
}

game.reset()
while not game.game_over():
    game.render()
    agent = agents[game.agent_selection]
    val = game.eval(agent.agent)
    action = agent.action()

    print(f"Turn {game.steps} - Distance for agent {game.agent_selection}: {game.distance_to_terminal(agent.agent)} - Eval: {val} - Action {action}\n")
    
    game.step(action=action)

game.render()


0: ___ ___ ___ ___ ___ 
1: 0__ 0__ 0__ 0__ 0__ 
2: ___ ___ ___ ___ ___ 
3: ___ ___ ___ ___ ___ 
4: ___ ___ ___ ___ ___ 
5: ___ ___ ___ ___ ___ 
6: 1__ 1__ 1__ 1__ 1__ 
7: ___ ___ ___ ___ ___ 
Turn 0 - Distance for agent Black: 6 - Eval: 0 - Action 52

0: ___ ___ ___ ___ ___ 
1: 0__ ___ 0__ 0__ 0__ 
2: 0__ ___ ___ ___ ___ 
3: ___ ___ ___ ___ ___ 
4: ___ ___ ___ ___ ___ 
5: ___ ___ ___ ___ ___ 
6: 1__ 1__ 1__ 1__ 1__ 
7: ___ ___ ___ ___ ___ 
Turn 1 - Distance for agent White: 6 - Eval: -1 - Action 263

0: ___ ___ ___ ___ ___ 
1: 0__ ___ 0__ 0__ 0__ 
2: 0__ ___ ___ ___ ___ 
3: ___ ___ ___ ___ ___ 
4: ___ ___ ___ ___ ___ 
5: ___ ___ ___ 1__ ___ 
6: 1__ 1__ ___ 1__ 1__ 
7: ___ ___ ___ ___ ___ 
Turn 2 - Distance for agent Black: 5 - Eval: 0 - Action 81

0: ___ ___ ___ ___ ___ 
1: 0__ ___ 0__ 0__ 0__ 
2: ___ ___ ___ ___ ___ 
3: 0__ ___ ___ ___ ___ 
4: ___ ___ ___ ___ ___ 
5: ___ ___ ___ 1__ ___ 
6: 1__ 1__ ___ 1__ 1__ 
7: ___ ___ ___ ___ ___ 
Turn 3 - Distance for agent White: 5 - Eval: -1 - 

Agregando esta simple función de evaluación se puede apreciar un comportamiento mucho más inteligente de los agentes que implementan minimax. Se puede ver como ahora los agentes tienden a avanzar sus fichas hacia el lado contrario del tablero, e incluso tienden a bloquear las fichas del otro agente con el objetivo de minimizar la distancia del rival a la meta.

En reiteradas iteraciones del juego se observa que una vez bloqueada la pieza de un rival que se encuentra próxima a ganar, el bloqueo se mantiene indefinidamente, no permitiendo al rival avanzar y desplegando otras piezas para buscar la victoria.

También se aprecia que los agentes con mayor profundidad `depth` tienden a ganar por sobre los agentes con menor profunidad. Se pudo observar que el aumento en la profundidad de exploración permite a los agentes encontrar caminos que se alejan de las fichas del rival cuando se encuentra cerca de la meta (escapando de un bloqueo) mientras que persiguen y bloquean a las fichas del rival cuando estas se encuentran cerca de su objetivo (buscando bloquearlas).

Sin embargo, la demora en la toma de acciones crece exponencialmente a medida que se aumenta el valor de `depth`, imposibilitando la experimentación con valores mayores a 5.

Probaremos ahora el desempeño de MCTS para este tipo de juegos.